In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, row_number, when
from pyspark.sql.window import Window
import numpy as np
import pandas as pd
import json

spark = SparkSession.builder \
    .appName("pertemuan7") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi spark: ", spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/09 00:18:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession siap. Versi spark:  3.5.9


In [2]:
np.random.seed(77)

# SUMBER 1: Data toko/cabang(JSON) - tabel referensi kecil
toko_list = ["Toko Magelang", "Toko Yogyakarta", "Toko Semarang", "Toko Solo"]
data_toko = [
    {"toko_id": i + 1, "nama_toko": t, "kota": t.split()[1], "manager": np.random.choice(["Rani", "Joko", "Sari", "Bayu"])}
    for i, t in enumerate(toko_list)
]
with open("data_toko.json", "w") as f:
    for t in data_toko:
        f.write(json.dumps(t) + "\n")

# SUMBER 2: Data produk (CSV) - tabel referensi kecil
data_produk = [
    {"produk_id": i, "nama_produk": f"Produk-{i}",
     "kategori": np.random.choice(["Elektronik", "Fashion", "Makanan"]),
     "harga": int(np.random.choice([50000, 100000, 200000]))}
    for i in range(1, 21)
]
pd.DataFrame(data_produk).to_csv("data_produk.csv", index=False)

# SUMBER 3: Data transaksi triwulan (CSV) - data utama, jauh lebih besar
n = 5000
data_transaksi = pd.DataFrame({
    "trx_id": [f"T{i}" for i in range(n)],
    "toko_id": np.random.randint(1, 5, size=n),
    "product_id": np.random.randint(1, 21, size=n),
    "qty": np.random.randint(1, 6, size=n),
})
data_transaksi.to_csv("data_transaksi_triwulan.csv", index=False)

print("Ketiga sumber data berhasil dibuat:")
print(f"- data_toko.json {len(data_toko)} baris")
print(f"- data_produk.csv {len(data_produk)} baris")
print(f"- data_transaksi_triwulan.csv {len(data_transaksi)} baris")

Ketiga sumber data berhasil dibuat:
- data_toko.json 4 baris
- data_produk.csv 20 baris
- data_transaksi_triwulan.csv 5000 baris


In [3]:
# Unggah sumber data utama ke HDFS
!hdfs dfs -mkdir -p /user/azka/pertemuan7/raw
!hdfs dfs -put -f data_transaksi_triwulan.csv /user/azka/pertemuan7/raw
!hdfs dfs -put -f data_toko.json /user/azka/pertemuan7/raw
!hdfs dfs -put -f data_produk.csv /user/azka/pertemuan7/raw
!hdfs dfs -ls /user/azka/pertemuan7/raw

Found 3 items
-rw-r--r--   1 azka supergroup        598 2026-10-09 00:20 /user/azka/pertemuan7/raw/data_produk.csv
-rw-r--r--   1 azka supergroup        332 2026-10-09 00:20 /user/azka/pertemuan7/raw/data_toko.json
-rw-r--r--   1 azka supergroup      61655 2026-10-09 00:20 /user/azka/pertemuan7/raw/data_transaksi_triwulan.csv


In [4]:
# Tahap extract membaca dari hdfs

df_transaksi = spark.read.csv("hdfs://localhost:9000/user/azka/pertemuan7/raw/data_transaksi_triwulan.csv", header=True, inferSchema=True)
df_toko = spark.read.json("hdfs://localhost:9000/user/azka/pertemuan7/raw/data_toko.json")
df_produk = spark.read.csv("hdfs://localhost:9000/user/azka/pertemuan7/raw/data_produk.csv", header=True, inferSchema=True)

print("Transaksi: ", df_transaksi.count(), "I Toko: ", df_toko.count(), "I Produk: ", df_produk.count())

Transaksi:  5000 I Toko:  4 I Produk:  20


In [5]:
# Tahap transform, menggabungkan 3 sumber dengan join

df_produk = df_produk.withColumnRenamed("produk_id", "product_id")

df_gabungan = df_transaksi.join(df_toko, on="toko_id", how="inner") \
                          .join(df_produk, on="product_id", how="inner")
df_gabungan = df_gabungan.withColumn("total_jualan", col("qty") * col("harga"))
df_gabungan.show(5)

+----------+-------+------+---+----------+-------+---------------+-----------+----------+------+------------+
|product_id|toko_id|trx_id|qty|      kota|manager|      nama_toko|nama_produk|  kategori| harga|total_jualan|
+----------+-------+------+---+----------+-------+---------------+-----------+----------+------+------------+
|         7|      1|    T0|  5|  Magelang|   Bayu|  Toko Magelang|   Produk-7|   Makanan|200000|     1000000|
|         5|      2|    T1|  5|Yogyakarta|   Bayu|Toko Yogyakarta|   Produk-5|Elektronik|200000|     1000000|
|         2|      3|    T2|  2|  Semarang|   Rani|  Toko Semarang|   Produk-2|Elektronik|100000|      200000|
|         2|      2|    T3|  2|Yogyakarta|   Bayu|Toko Yogyakarta|   Produk-2|Elektronik|100000|      200000|
|        18|      4|    T4|  2|      Solo|   Rani|      Toko Solo|  Produk-18|Elektronik|100000|      200000|
+----------+-------+------+---+----------+-------+---------------+-----------+----------+------+------------+
only showi

In [6]:
# Window function, top 3 produk terlaris per toko

# Total penjualan per produk di tiap toko
ringkasan_produk_toko = (
    df_gabungan
    .groupBy("nama_toko", "nama_produk")
    .agg(spark_sum("total_jualan").alias("total_penjualan"))
)

# Window function untuk top 3 per toko
window_toko = Window.partitionBy("nama_toko").orderBy(col("total_penjualan").desc())

top3_per_toko = (
    ringkasan_produk_toko
    .withColumn("peringkat", row_number().over(window_toko))
    .filter(col("peringkat") <= 3)
    .orderBy("nama_toko", "peringkat")
)

top3_per_toko.show(20)

[Stage 21:>                                                         (0 + 1) / 1]

+---------------+-----------+---------------+---------+
|      nama_toko|nama_produk|total_penjualan|peringkat|
+---------------+-----------+---------------+---------+
|  Toko Magelang|   Produk-7|       46000000|        1|
|  Toko Magelang|   Produk-5|       44400000|        2|
|  Toko Magelang|  Produk-14|       38400000|        3|
|  Toko Semarang|   Produk-8|       49200000|        1|
|  Toko Semarang|   Produk-7|       40400000|        2|
|  Toko Semarang|   Produk-5|       39000000|        3|
|      Toko Solo|  Produk-14|       38400000|        1|
|      Toko Solo|   Produk-7|       37200000|        2|
|      Toko Solo|   Produk-5|       33200000|        3|
|Toko Yogyakarta|  Produk-14|       48200000|        1|
|Toko Yogyakarta|   Produk-5|       37600000|        2|
|Toko Yogyakarta|   Produk-4|       37000000|        3|
+---------------+-----------+---------------+---------+



In [7]:
# Kontribusi Manajer Per Cabang 

df_gabungan.createOrReplaceTempView("gabungan")

kontribusi_manager = spark.sql('''
    SELECT kota, manager, SUM(total_jualan) AS total_penjualan
    FROM gabungan
    GROUP BY kota, manager
    ORDER BY total_penjualan DESC
''')
kontribusi_manager.show()

+----------+-------+---------------+
|      kota|manager|total_penjualan|
+----------+-------+---------------+
|  Semarang|   Rani|      416250000|
|  Magelang|   Bayu|      408550000|
|Yogyakarta|   Bayu|      389150000|
|      Solo|   Rani|      369950000|
+----------+-------+---------------+



In [8]:
# menyimpan laporan ke hdfs

!hdfs dfs -mkdir -p /user/mahasiswa/pertemuan7/processed

df_gabungan.write.mode("overwrite").partitionBy("kategori").parquet(
    "hdfs://localhost:9000/user/mahasiswa/pertemuan7/processed/laporan_triwulan"
)

print("Laporan berhasil disimpan ke HDFS dalam format Parquet, terpartisi per kategori produk.")
!hdfs dfs -ls /user/mahasiswa/pertemuan7/processed/laporan_triwulan

Laporan berhasil disimpan ke HDFS dalam format Parquet, terpartisi per kategori produk.
Found 4 items
-rw-r--r--   3 azka supergroup          0 2026-10-09 00:53 /user/mahasiswa/pertemuan7/processed/laporan_triwulan/_SUCCESS
drwxr-xr-x   - azka supergroup          0 2026-10-09 00:53 /user/mahasiswa/pertemuan7/processed/laporan_triwulan/kategori=Elektronik
drwxr-xr-x   - azka supergroup          0 2026-10-09 00:53 /user/mahasiswa/pertemuan7/processed/laporan_triwulan/kategori=Fashion
drwxr-xr-x   - azka supergroup          0 2026-10-09 00:53 /user/mahasiswa/pertemuan7/processed/laporan_triwulan/kategori=Makanan


In [9]:
# Verifikasi akhir

df_final = spark.read.parquet("hdfs://localhost:9000/user/mahasiswa/pertemuan7/processed/laporan_triwulan")

ringkasan_eksekutif = df_final.groupBy("nama_toko", "kota", "manager").agg(
    spark_sum("total_jualan").alias("total_penjualan_triwulan"),
    count("trx_id").alias("jumlah_transaksi"),
    avg("total_jualan").alias("rata_rata_nilai_transaksi")
).orderBy(col("total_penjualan_triwulan").desc())

ringkasan_eksekutif.show()

[Stage 34:>                                                         (0 + 3) / 3]

+---------------+----------+-------+------------------------+----------------+-------------------------+
|      nama_toko|      kota|manager|total_penjualan_triwulan|jumlah_transaksi|rata_rata_nilai_transaksi|
+---------------+----------+-------+------------------------+----------------+-------------------------+
|  Toko Semarang|  Semarang|   Rani|               416250000|            1306|       318721.28637059726|
|  Toko Magelang|  Magelang|   Bayu|               408550000|            1315|       310684.41064638784|
|Toko Yogyakarta|Yogyakarta|   Bayu|               389150000|            1200|        324291.6666666667|
|      Toko Solo|      Solo|   Rani|               369950000|            1179|        313782.8668363019|
+---------------+----------+-------+------------------------+----------------+-------------------------+



In [10]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
